# Phase 3: nothing is independent

Three layers, each building on the last (`docs/decisions/0003`, `0007`; `docs/math.md` 5-6):

1. **Pairwise dependence.** For every cross-group pair (fiscal x rates x macro x labor): full-sample and rolling
   correlation, lead/lag scan, Granger causality both ways with Benjamini-Hochberg control. A wide rolling range means
   one number misdescribes the pair.
2. **Block factors.** Highly correlated series act as one: each behavioral block (consumer, corporate, financial
   conditions, fiscal, labor, rates, prices) is reduced to a single interpretable factor. The residual of each series
   against its factor is the imperfect-correlation shock.
3. **Latent states.** A Markov chain on the factor vector, estimated as a hidden Markov model, with within-state
   dynamics and bootstrapped residuals. Outcomes become mixtures over states, not normals.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
import bond_sim.notebook as nb
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
plt.rcParams["figure.dpi"] = 110

# ── parameters: change and rerun ─────────────────────────────────────────────
AS_OF = pd.Timestamp("2026-09-15")     # every loader filters on this date (point-in-time contract)
K = 1000                                 # Monte Carlo paths (raise for the paper)
FORCE_REFRESH = False                  # True re-downloads every series and rebuilds cached steps
os.environ["BOND_SIM_CACHE"] = "0" if FORCE_REFRESH else "1"
import bond_sim.config as bcfg
CFG_HASH = bcfg.load().content_hash()   # every cached step is keyed by the configuration that produced it
ctx = nb.cached(f"ctx_{AS_OF.date()}_{CFG_HASH}", lambda: nb.load_context(AS_OF), refresh=FORCE_REFRESH)
print(ctx.grid, "| config hash", CFG_HASH, "| series:", ctx.w.shape[1])

## 3.1 Pairwise dependence on the thesis pairs

In [ ]:
from bond_sim.analysis import CorrelationAnalyzer, RegimeDetector
from bond_sim.cli import THESIS_GROUPS, _quarterly_panel
ids = [s for g in THESIS_GROUPS.values() for s in g]
group_of = {s: g for g, ss in THESIS_GROUPS.items() for s in ss}
wq = _quarterly_panel(ctx.w[[c for c in ids if c in ctx.w.columns]], ids)
an = CorrelationAnalyzer(wq, ctx.cfg.correlation, freq="Q")
rep = nb.cached(f"corr_report_{AS_OF.date()}_{CFG_HASH}", lambda: an.report(with_granger=True, group_of=group_of), refresh=FORCE_REFRESH)
display(rep.head(15)[["a", "b", "n", "full_corr", "rolling_min", "rolling_max", "best_lag", "leads", "granger_a_to_b_sig", "granger_b_to_a_sig"]].round(3))
print(f"{len(rep)} cross-group pairs; regime-dependent (rolling range > 0.5): {int(rep.regime_dependent.sum())}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
for row, pair in zip(axes, [("FYGFGDQ188S", "DGS10"), ("FYGFGDQ188S", "THREEFYTP10")]):
    r = an.analyze_pair(*pair, with_granger=False)
    r.rolling.plot(ax=row[0]); row[0].axhline(0, color="k", lw=0.8); row[0].set_title(f"20q rolling corr: d({pair[0]}) vs d({pair[1]})"); row[0].set_xlabel("")
    r.lead_lag.plot.bar(ax=row[1]); row[1].set_title(f"corr(d{pair[0]}_t, d{pair[1]}_(t+lag)); +lag = debt leads")
plt.tight_layout(); plt.show()
det = RegimeDetector(ctx.cfg.correlation)
top = rep[rep.regime_dependent].head(12)
labels = {f"{x.a}|{x.b}": det.label(an.rolling(x.a, x.b).dropna()) for x in top.itertuples()}
cons = det.consistency(labels); off = cons.to_numpy()[~np.eye(len(cons), dtype=bool)]
print(f"regime timing consistency across the 12 widest pairs, mean adjusted Rand index: {np.nanmean(off):.3f}  (near 0: pair-specific; near 1: one market regime)")

## 3.2 Block factors

One factor per block, sign-aligned so that high means expansionary / loose / strong / healthy. The loadings say which
series carry the block; the explained variance says how much of the block is one thing. Sample: the common overlap of all
seven blocks (VIX and the SLOOS series begin 1990).

In [ ]:
bf, chain = nb.cached(f"factors_chain_{AS_OF.date()}_{CFG_HASH}", lambda: nb.fit_factors_and_chain(ctx), refresh=FORCE_REFRESH)
display(bf.summary())
load = pd.DataFrame({b: {c: round(float(l), 2) for c, l in zip(f.columns, f.loadings)} for b, f in bf.fits.items()})
display(load.fillna("").T)
F = bf.factors()
fig, axes = plt.subplots(len(F.columns), 1, figsize=(13, 2.0 * len(F.columns)), sharex=True)
for ax, b in zip(axes, F.columns):
    F[b].plot(ax=ax, lw=0.9); ax.axhline(0, color="k", lw=0.5); ax.set_ylabel(b)
    for t0, t1 in [("1990-07-01", "1991-03-01"), ("2001-03-01", "2001-11-01"), ("2007-12-01", "2009-06-01"), ("2020-02-01", "2020-04-01")]:
        ax.axvspan(pd.Timestamp(t0), pd.Timestamp(t1), color="grey", alpha=0.15)
axes[0].set_title("Block factors (grey: NBER recessions)"); plt.tight_layout(); plt.show()
display(F.corr().round(2))

## 3.3 The latent chain

A Gaussian HMM on the factor vector; the state count by BIC over 2-4; a sticky Dirichlet prior on the transition
matrix (sparse regime data, the RateWalk lesson); state 0 is always the weakest by construction. Within each state a
common VAR(1) with state-specific intercepts gives the dynamics; residuals are banked by state for bootstrapping.
The stress hazard is logistic in the financial-conditions factor: the doom-loop channel in state space.

In [ ]:
display(pd.DataFrame(chain.selection).T.rename_axis("candidate states"))
print(f"chosen: {chain.k} states (lowest BIC among candidates whose every state has >= {chain.min_state_occupancy} quarters; factors winsorized at +/-{chain.winsor_sd} sd for estimation)")
print(f"hazard beta (stress entry on the financial factor): {chain.hazard_beta:.3f} (unrestricted estimate {chain.hazard_beta_raw:.3f}; negative estimates disable the channel)   max eigenvalue of A: {np.abs(np.linalg.eigvals(chain.A)).max():.3f}")
display(pd.DataFrame(chain.P, index=[f"from {s}" for s in range(chain.k)], columns=[f"to {s}" for s in range(chain.k)]).round(3))
display(pd.DataFrame(chain.means, columns=chain.blocks, index=[f"state {s}" for s in range(chain.k)]).round(2))
fig, ax = plt.subplots(figsize=(13, 3))
chain.posterior.plot.area(ax=ax, linewidth=0, alpha=0.8); ax.set_title("Smoothed state probabilities"); ax.set_xlabel(""); plt.show()
print("state counts:", np.bincount(chain.states, minlength=chain.k).tolist())
print("quarters in state 0:", [d.strftime("%Y-%m") for d in chain.states[chain.states == 0].index])

## 3.4 Why not a normal distribution

The residuals the simulator bootstraps from are not Gaussian: skewness and excess kurtosis by state, against zero for
a normal. This is the empirical basis for drawing from history instead of from a fitted family.

In [ ]:
from scipy import stats
rows = []
for s, bank in chain.resid_bank.items():
    for j, b in enumerate(chain.blocks):
        rows.append({"state": s, "block": b, "n": len(bank), "skew": stats.skew(bank[:, j]), "excess_kurtosis": stats.kurtosis(bank[:, j])})
display(pd.DataFrame(rows).pivot(index="block", columns="state", values=["skew", "excess_kurtosis"]).round(2))
draws = chain.draw(2000, 120, np.random.default_rng(0))
sim = chain.simulate(2000, 120, draws, bf.last_factors(), chain.initial_state_probs())
occ = chain.occupancy(sim["S"])
ax = occ.plot.area(figsize=(11, 3), linewidth=0, alpha=0.8); ax.set_title("Unconditional state occupancy over 30 years, 2000 paths"); ax.set_xlabel("quarter"); plt.show()